In [52]:
import pandas as pd
df =pd.read_excel("./1.xlsx",encoding='gbk')

In [53]:
#keywords=['26283121778226','26076662786226','26067223087226','25779902720226','26066979183226','25785921087226','26066692527226','26067006475226']
 #10000229167469,10000229407532,10000265046381,10000270409428,10000269553640,10000273809516
#df['商品ID'].isin(keywords)
#df['商品ID'] = df['商品ID'].astype(str)


#删除其他行
#df.drop(index= df[ df['商品ID'].isin(keywords)==False].index,inplace=True)
#df.reset_index(drop=True,inplace= True)

In [54]:
df["实付款"] = df["实付款"].astype(str).str.replace("¥", "").astype(float)

In [55]:
from datetime import datetime
df["订单创建时间"] = pd.to_datetime(df["订单创建时间"],format = "%Y-%m-%d %H:%M:%S")
df["订单创建时间"] = df["订单创建时间"].map(lambda x: x.strftime('%Y-%m-%d'))
df1 = pd.read_excel("商品编码.xlsx")
df1.rename(columns={"商品名":"产品名称","商家编码":"SKU编码"},inplace=True)
df1 = df1[["SKU编码","产品名称"]]
df= pd.merge(df,df1,on="SKU编码",how="left")

In [56]:
# 缺码表
lackCode = df[df['产品名称'].isnull()]['SKU编码'].value_counts()
lackCode

Y117灰咖M    1
Name: SKU编码, dtype: int64

In [57]:
lackCode.to_excel('缺码表.xlsx')

In [58]:
df['订单数'] = df['成交数量']

In [59]:
df["下单单量"] = df['订单数']

In [60]:
# 清除未支付订单
from datetime import datetime
df["订单支付时间"] = df["订单支付时间"].fillna(pd.to_datetime('1970-01-01'))
df["订单支付时间"] = pd.to_datetime(df["订单支付时间"],format = "%Y-%m-%d %H:%M:%S")
df["订单支付时间"] = df["订单支付时间"].map(lambda x: x.strftime('%Y-%m-%d'))

In [61]:
df["发货时间"] = df["发货时间"].fillna(pd.to_datetime('1970-01-01'))
df["发货时间"] = pd.to_datetime(df["发货时间"],format = "%Y-%m-%d %H:%M:%S")
df["发货时间"] = df["发货时间"].map(lambda x: x.strftime('%Y-%m-%d'))

In [62]:
df["支付单量"] = 0
df["发货单量"] = 0
df["支付金额"] = 0
for i in range(0,len(df)):
    if df.loc[i,"发货时间"] != "1970-01-01":
        df.loc[i,"发货单量"] = df.loc[i,'订单数']
for i in range(0,len(df)):
    if df.loc[i,"订单支付时间"] != "1970-01-01":
        df.loc[i,"支付单量"] = df.loc[i,'订单数']
        df.loc[i,"支付金额"] = df.loc[i,'实付款']

In [63]:
df["售后状态"] = df["售后状态"].astype(str)

In [64]:
df["商品ID"] = df["商品ID"].astype(str)

In [65]:
df["售后状态"].value_counts()

nan      16624
退款成功      5063
待卖家收货      746
待买家退货      197
退款关闭       114
待卖家处理       13
待买家收货        7
换货关闭         4
待买家处理        4
换货成功         1
Name: 售后状态, dtype: int64

In [66]:
df["商品ID"].value_counts()

25290697648714    19795
27110313506714     1744
27017417147714      875
27143920832714       88
27218619573714       86
27207700941714       56
27017598879714       41
27143760567714       29
25310693452714       24
27009686396714       15
27143799053714        5
27198276921714        5
27198225385714        4
27082326616714        2
27143760726714        2
24266175376714        1
27210186735714        1
Name: 商品ID, dtype: int64

In [67]:
df['物流信息'].value_counts()

已签收       13002
未知         3910
运输中        1433
商家已接单      1297
已发货        1239
待取件         935
已退签         521
派送中         268
退回中         138
中转仓处理中       19
收件人拒签         6
商家拣货中         3
已揽件           1
已出中转仓         1
Name: 物流信息, dtype: int64

In [68]:
for i in range (0,len(df)):
    if df.loc[i,"订单状态"] == "已收货" and (df.loc[i,'售后状态'] == 'nan' or df.loc[i,'售后状态'] == '退款关闭'):
        df.loc[i,"订单状态"] = "交易成功"

In [69]:
df["已签收"] = 0
df["发货中"] = 0
df["售后中"] = 0
df["已退款"] = 0
df["预计签收数量"] = 0
df["发货前退款单量"] = 0
for i in range(0,len(df)):
    if df.loc[i,"订单状态"] == "交易成功" and (df.loc[i,"售后状态"]== "nan" or df.loc[i,"售后状态"]=="退款关闭"):
        df.loc[i,"已签收"] = df.loc[i,"订单数"]
for i in range(0,len(df)):
    if df.loc[i,"订单状态"] == "已发货" and (df.loc[i,"售后状态"]== "nan" or df.loc[i,"售后状态"]=="退款关闭"):
        df.loc[i,"发货中"] = df.loc[i,"订单数"]
for i in range(0,len(df)):
    if df.loc[i,"售后状态"] == "待买家处理" or df.loc[i,"售后状态"] == "待买家退货" or df.loc[i,"售后状态"] == "待卖家处理" or df.loc[i,"售后状态"]=="待卖家收货" or df.loc[i,"售后状态"]=="待平台判定":
        df.loc[i,"售后中"] = df.loc[i,"订单数"]
for i in range(0,len(df)):
    if df.loc[i,"发货时间"] != "1970-01-01" and df.loc[i,"售后状态"] == "退款成功":
        df.loc[i,"已退款"] = df.loc[i,'订单数']
for i in range(0,len(df)):
    if df.loc[i,"订单支付时间"] != "1970-01-01"  and  df.loc[i,"发货时间"] == "1970-01-01" and df.loc[i,"售后状态"] == "退款成功":
        df.loc[i,"发货前退款单量"] = df.loc[i,'订单数']
for i in range(0,len(df)):
    if df.loc[i,"发货时间"] != "1970-01-01" and (df.loc[i,"售后状态"]== "nan" or df.loc[i,"售后状态"]=="退款关闭"):
        df.loc[i,"预计签收单量"] = df.loc[i,"订单数"]

In [70]:
import numpy as np
result = pd.pivot_table(df,index=['订单创建时间',"产品名称"] , values = ['下单单量','支付单量','发货单量','已签收','发货中','售后中','已退款','发货前退款单量',"预计签收单量",'支付金额'] , aggfunc = np.sum)
result = result.reset_index()
result = result[['订单创建时间',"产品名称",'下单单量','支付单量','发货单量','已签收','发货中','售后中','已退款','发货前退款单量',"预计签收单量",'支付金额']]

In [71]:
result.to_excel("111.xlsx",index=False)

In [72]:
# 拼接店铺消耗表
df_consume = pd.read_excel("快手店铺消耗10号.xlsx")
df_consume = df_consume[['日期','产品','花费','转化数']]
temp = df_consume.日期.str.extract('(\d+).*?(\d+).*?(\d+)') 
df_consume['订单创建时间'] = temp[0] + "-" + temp[1] + '-' + temp[2]
df_consume.rename(columns={"产品":"产品名称"},inplace=True)
del df_consume['日期']
df_consume = pd.pivot_table(df_consume,index=['订单创建时间','产品名称'] , values = ['花费','转化数'] , aggfunc = np.sum)
df_consume = df_consume.reset_index()
df_consume["订单创建时间"] = pd.to_datetime(df_consume["订单创建时间"],format = "%Y-%m-%d")
df_consume["订单创建时间"] = df_consume["订单创建时间"].map(lambda x: x.strftime('%Y-%m-%d'))

In [73]:
final = pd.merge(result,df_consume,on = ['订单创建时间','产品名称'],how = 'left')
final = final[['订单创建时间',"产品名称",'转化数','下单单量','支付单量','发货单量','已签收','发货中','售后中','已退款','发货前退款单量',"预计签收单量",'支付金额']]
final.rename(columns={"转化数":"总广告单量"},inplace=True)

In [74]:
final

,订单创建时间,产品名称,总广告单量,下单单量,支付单量,发货单量,已签收,发货中,售后中,已退款,发货前退款单量,预计签收单量,支付金额
0,2026-10-29,M68保暖背心,507.0,577,511,477,316,0,1,160,34,316.0,41195.3
1,2026-10-30,M68保暖背心,437.0,518,441,408,247,0,1,160,33,247.0,35425.0
2,2026-10-31,M68保暖背心,397.0,484,403,369,213,2,2,152,34,215.0,32397.0
3,2026-11-01,M68保暖背心,203.0,290,243,226,141,0,0,85,17,141.0,19485.7
4,2026-11-02,M68保暖背心,1050.0,1245,1056,975,639,2,6,328,81,641.0,84959.0
5,2026-11-03,M68保暖背心,958.0,1117,961,878,598,2,5,273,83,600.0,77003.0
6,2026-11-04,M68保暖背心,484.0,601,485,437,324,1,3,109,48,325.0,38901.5
7,2026-11-05,M68保暖背心,290.0,354,299,281,188,0,4,89,18,188.0,24077.4
8,2026-11-06,M68保暖背心,524.0,640,534,488,311,3,5,169,46,314.0,42865.7
9,2026-11-07,M68保暖背心,850.0,1015,861,778,541,3,17,217,83,544.0,69243.9


In [75]:
final.to_excel('直播签收汇总.xlsx')